In [1]:
from google.colab import files
!pip -q install kaggle gdown scipy scikit-learn matplotlib tensorboard tqdm pillow opencv-python-headless
from google.colab import files

kaggle_json = "/content/drive/MydDrive/kaggle.json"
print("Kaggle API ready:", kaggle_json)

Kaggle API ready: /content/drive/MydDrive/kaggle.json


In [2]:
!kaggle datasets download nikanvasei/shanghaitech-campus-dataset-test

Dataset URL: https://www.kaggle.com/datasets/nikanvasei/shanghaitech-campus-dataset-test
License(s): MIT
100% 8.72G/8.72G [01:34<00:00, 99.2MB/s]



In [3]:
%%bash
# install 7z library
wget https://www.7-zip.org/a/7z2301-linux-x64.tar.xz
tar -xf /content/7z2301-linux-x64.tar.xz

--2026-05-08 21:18:34--  https://www.7-zip.org/a/7z2301-linux-x64.tar.xz
Resolving www.7-zip.org (www.7-zip.org)... 89.167.91.206
Connecting to www.7-zip.org (www.7-zip.org)|89.167.91.206|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1527700 (1.5M) [application/octet-stream]
Saving to: ‘7z2301-linux-x64.tar.xz’

     0K .......... .......... .......... .......... ..........  3%  308K 5s
    50K .......... .......... .......... .......... ..........  6%  308K 5s
   100K .......... .......... .......... .......... .......... 10%  115M 3s
   150K .......... .......... .......... .......... .......... 13% 1.24M 2s
   200K .......... .......... .......... .......... .......... 16%  407K 2s
   250K .......... .......... .......... .......... .......... 20%  109M 2s
   300K .......... .......... .......... .......... .......... 23% 93.3M 2s
   350K .......... .......... .......... .......... .......... 26% 1.26M 1s
   400K .......... .......... .......... .........

In [4]:
!/content/7zz x /content/shanghaitech-campus-dataset-test.zip


7-Zip (z) 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576, ASM

Scanning the drive for archives:
  0M Scan /content                  1 file, 9364155814 bytes (8931 MiB)

Extracting archive: /content/shanghaitech-campus-dataset-test.zip
  2% 4096 Op            --
Path = /content/shanghaitech-campus-dataset-test.zip
Type = zip
Physical Size = 9364155814
64-bit = +
Characteristics = Zip64

      0% 3        0% 7        0% 9        0% 112          0% 1398 - SHANGHAI/SHANGHAI_Test/frames/01_0025/199.jp                                                          1% 165          1% 184          1% 207          1% 234          2% 265          2% 337          2% 367          2% 387          2% 414          3% 441          3% 461          3% 491          4% 520          4% 559          4% 601          4% 619          4% 644          4% 688          4% 727          5% 752          5% 778          5% 800          5% 817          5%

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import json
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import Video

# Change these
clip_dir = Path("/content/SHANGHAI/SHANGHAI_Test/frames/01_0028")
json_path = Path("/content/drive/MyDrive/STG-NF/kaggle_shanghai/pose/test/01_0028_alphapose-results.json")
out_path = Path("/content/01_0028_pose_video.mp4")

# COCO-17 skeleton
SKELETON = [
    (0,1), (0,2), (1,3), (2,4),
    (5,6), (5,7), (7,9), (6,8), (8,10),
    (5,11), (6,12), (11,12),
    (11,13), (13,15), (12,14), (14,16)
]

def natural_key(p):
    stem = p.stem
    return int(stem) if stem.isdigit() else stem

def color_from_id(idx):
    idx = int(idx)
    return (
        (37 * idx) % 255,
        (97 * idx) % 255,
        (173 * idx) % 255,
    )

# Load AlphaPose rows
rows = json.loads(json_path.read_text())

# Group detections by frame image
by_image = defaultdict(list)
for row in rows:
    by_image[Path(row["image_id"]).name].append(row)

# Read ordered frame list
frame_files = sorted(
    [p for p in clip_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]],
    key=natural_key
)

# Init video writer
first = cv2.imread(str(frame_files[0]))
h, w = first.shape[:2]
writer = cv2.VideoWriter(
    str(out_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    25,
    (w, h)
)

KP_THR = 0.25

for frame_path in frame_files:
    img = cv2.imread(str(frame_path))
    image_name = frame_path.name

    for det in by_image.get(image_name, []):
        kp = np.array(det["keypoints"], dtype=np.float32).reshape(-1, 3)
        bbox = det.get("box", None)
        track_id = int(det.get("idx", -1))
        color = color_from_id(track_id if track_id >= 0 else 0)

        # draw bbox
        if bbox is not None and len(bbox) >= 4:
            x, y, bw, bh = bbox[:4]
            x1, y1 = int(x), int(y)
            x2, y2 = int(x + bw), int(y + bh)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                img,
                f"ID {track_id}",
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

        # draw joints
        for x, y, c in kp:
            if c >= KP_THR:
                cv2.circle(img, (int(x), int(y)), 3, color, -1)

        # draw skeleton
        for a, b in SKELETON:
            if kp[a, 2] >= KP_THR and kp[b, 2] >= KP_THR:
                pt1 = (int(kp[a, 0]), int(kp[a, 1]))
                pt2 = (int(kp[b, 0]), int(kp[b, 1]))
                cv2.line(img, pt1, pt2, color, 2)

    writer.write(img)

writer.release()
print("Saved:", out_path)


Saved: /content/01_0028_pose_video.mp4
